<a href="https://colab.research.google.com/github/Karthikreddy1010/Microplastic_Image_Segmentation/blob/main/microplastic_particle_count.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!unzip /content/Full_Beads_Microplastic.zip

Archive:  /content/Full_Beads_Microplastic.zip
   creating: Full_Beads_Microplastic/
   creating: Full_Beads_Microplastic/Testing/
   creating: Full_Beads_Microplastic/Testing/Testing/
   creating: Full_Beads_Microplastic/Testing/Testing/Original/
  inflating: Full_Beads_Microplastic/Testing/Testing/Original/a--23-_jpg.rf.1ab5e302030f3bb3c08981ca42a8e631.jpg  
  inflating: Full_Beads_Microplastic/Testing/Testing/Original/a--26-_jpg.rf.0ae749f9f22dbfa00f0889c68594bdc9.jpg  
  inflating: Full_Beads_Microplastic/Testing/Testing/Original/a--27-_jpg.rf.8a800e6ec700f849fa4235fb736d6532.jpg  
  inflating: Full_Beads_Microplastic/Testing/Testing/Original/a--3-_jpg.rf.8248ba99e3b3ae254d1723b674f7fd99.jpg  
  inflating: Full_Beads_Microplastic/Testing/Testing/Original/a--4-_jpg.rf.11d39a37cd66634e646df4be915b9044.jpg  
  inflating: Full_Beads_Microplastic/Testing/Testing/Original/a--43-_jpg.rf.feae28fb9f182a3b7cfd06be68c8bef3.jpg  
  inflating: Full_Beads_Microplastic/Testing/Testing/Original/a-

In [11]:
import cv2
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats as sp_stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
import json
from skimage.segmentation import find_boundaries



class GTConfig:
    """
    Unified rules for ALL microplastics (beads + irregulars)
    Key principle: 1 connected region = 1 particle
    """
    # Size filtering (noise only)
    MIN_AREA = 15           # Remove tiny noise (adjust based on image resolution)
    MAX_AREA = 15000        # Remove large artifacts (very conservative)

    # Border exclusion (objective)
    BORDER_WIDTH = 8        # Remove partial/ambiguous particles

    # Connectivity (standard)
    CONNECTIVITY = 8        # Standard for object counting

    # NO shape filtering in GT
    MIN_CIRCULARITY = 0.0   # Zero = no circularity filtering in GT

    # Post-analysis thresholds (NOT used in GT counting)
    POST_CIRCULARITY_BEAD = 0.7      # For post-hoc classification
    POST_ASPECT_RATIO_FIBER = 3.0    # For post-hoc classification
    POST_SOLIDITY_IRREGULAR = 0.85   # For post-hoc classification

    SAVE_CLEANED_MASKS = True

    @classmethod
    def description(cls):
        """Paper-ready methodology description"""
        return (
            "Ground-truth particle counts for mixed microplastics (beads and irregular particles) "
            "were derived from annotated segmentation masks by identifying connected components. "
            f"Size ({cls.MIN_AREA}–{cls.MAX_AREA} pixels) and border ({cls.BORDER_WIDTH}-pixel exclusion) "
            "constraints were applied to remove noise and ambiguous edge particles. "
            "No shape-based filtering was applied at the ground-truth stage to maintain objectivity "
            "for irregular microplastics. Particle morphology was analyzed separately using "
            "post hoc shape descriptors."
        )



def preprocess_gt_mask(mask, image_idx=None, config=GTConfig):
    """
    Clean GT mask using UNIFIED rules for all microplastics
    """
    # Ensure 2D and binary
    if mask.ndim == 3:
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
    mask = mask.squeeze()
    mask = (mask > 0).astype(np.uint8)

    if mask.sum() == 0:
        return mask, []

    # Connected components
    num_labels, labels, comp_stats, centroids = cv2.connectedComponentsWithStats(
        mask, connectivity=config.CONNECTIVITY
    )

    cleaned = np.zeros_like(mask)
    component_data = []
    height, width = mask.shape

    for i in range(1, num_labels):
        area = comp_stats[i, cv2.CC_STAT_AREA]

        #  Size filtering only (no shape bias)
        if area < config.MIN_AREA or area > config.MAX_AREA:
            continue

        # Border exclusion (objective)
        x = comp_stats[i, cv2.CC_STAT_LEFT]
        y = comp_stats[i, cv2.CC_STAT_TOP]
        w = comp_stats[i, cv2.CC_STAT_WIDTH]
        h = comp_stats[i, cv2.CC_STAT_HEIGHT]

        if (x < config.BORDER_WIDTH or
            y < config.BORDER_WIDTH or
            x + w > width - config.BORDER_WIDTH or
            y + h > height - config.BORDER_WIDTH):
            continue

        #  Keep ALL valid components (beads AND irregulars)
        cleaned[labels == i] = 1

        # Compute shape metrics for POST-ANALYSIS (not filtering)
        shape_metrics = compute_shape_metrics(labels == i, area)

        component_data.append({
            'image_index': image_idx,  # Track image association
            'component_id': i,
            'area': int(area),  # Convert to Python int for JSON
            'centroid': [float(centroids[i][0]), float(centroids[i][1])],  # Convert to float
            'bbox': [int(x), int(y), int(w), int(h)],
            **shape_metrics  # For post-analysis
        })

    return cleaned, component_data

def compute_shape_metrics(component_mask, area):
    """
    Compute shape metrics for POST-HOC analysis only
     FIXED: Eccentricity calculation with safe sqrt
    """
    component_mask = component_mask.astype(np.uint8)
    metrics = {
        'circularity': None,
        'aspect_ratio': None,
        'solidity': None,
        'convex_hull_area': None,
        'perimeter': None,
        'eccentricity': None
    }

    # Find contours
    contours, _ = cv2.findContours(
        component_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if contours:
        cnt = contours[0]

        # Perimeter
        perimeter = cv2.arcLength(cnt, True)
        metrics['perimeter'] = float(perimeter)

        # Circularity
        if perimeter > 0:
            metrics['circularity'] = float(4 * np.pi * area / (perimeter ** 2))

        # Bounding rectangle for aspect ratio
        _, _, w, h = cv2.boundingRect(cnt)
        if min(w, h) > 0:
            metrics['aspect_ratio'] = float(max(w, h) / min(w, h))

        # Convex hull and solidity
        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        metrics['convex_hull_area'] = float(hull_area)
        if hull_area > 0:
            metrics['solidity'] = float(area / hull_area)

        # Fit ellipse for eccentricity -  FIXED with safe calculation
        if len(cnt) >= 5:
            try:
                (_, _), (MA, ma), angle = cv2.fitEllipse(cnt)
                if MA > 0 and ma > 0:
                    # Ensure MA >= ma and calculate safely
                    major_axis = max(MA, ma)
                    minor_axis = min(MA, ma)
                    ratio = (minor_axis ** 2) / (major_axis ** 2)
                    if ratio <= 1:  # Safety check
                        metrics['eccentricity'] = float(np.sqrt(1 - ratio))
                    else:
                        metrics['eccentricity'] = 0.0
                else:
                    metrics['eccentricity'] = None
            except Exception as e:
                metrics['eccentricity'] = None

    return metrics

def count_particles(mask):
    """Count connected components"""
    mask = mask.squeeze().astype(np.uint8)
    if mask.max() == 0:
        return 0
    num_labels, _ = cv2.connectedComponents(mask, connectivity=GTConfig.CONNECTIVITY)
    return num_labels - 1



class MorphologyAnalyzer:
    """Analyze particle morphology AFTER GT counting"""

    @staticmethod
    def classify_particles(component_data, config=GTConfig):
        """
        Classify particles by morphology (POST-HOC only)
        """
        if not component_data:
            return []

        classified = []

        for comp in component_data:
            # Default classification
            comp_class = "unclassified"

            # Classification rules (POST-HOC ONLY)
            circularity = comp.get('circularity', 0)
            aspect_ratio = comp.get('aspect_ratio', 1)
            solidity = comp.get('solidity', 1)

            # Rule 1: Bead-like (high circularity)
            if circularity is not None and circularity >= config.POST_CIRCULARITY_BEAD:
                comp_class = "bead-like"

            # Rule 2: Fiber-like (high aspect ratio)
            elif aspect_ratio is not None and aspect_ratio >= config.POST_ASPECT_RATIO_FIBER:
                comp_class = "fiber-like"

            # Rule 3: Irregular (low solidity)
            elif solidity is not None and solidity < config.POST_SOLIDITY_IRREGULAR:
                comp_class = "irregular"

            # Rule 4: Compact (everything else)
            else:
                comp_class = "compact"

            # Add classification to component data
            comp_with_class = comp.copy()
            comp_with_class.update({
                'morphology_class': comp_class,
                'is_bead_like': comp_class == "bead-like",
                'is_fiber_like': comp_class == "fiber-like",
                'is_irregular': comp_class == "irregular",
                'is_compact': comp_class == "compact"
            })
            classified.append(comp_with_class)

        return classified

    @staticmethod
    def analyze_dataset(dataset_name, all_component_data):
        """
        Generate comprehensive morphology analysis
        """
        if not all_component_data:
            return None

        # Group components by image
        images_dict = {}
        for comp in all_component_data:
            idx = comp.get('image_index', -1)
            if idx not in images_dict:
                images_dict[idx] = []
            images_dict[idx].append(comp)

        # Classify all particles
        all_classified = MorphologyAnalyzer.classify_particles(all_component_data)

        # Count by class
        class_counts = {}
        for comp in all_classified:
            cls = comp['morphology_class']
            class_counts[cls] = class_counts.get(cls, 0) + 1

        # Area statistics by class
        area_by_class = {}
        for cls in class_counts.keys():
            areas = [c['area'] for c in all_classified if c['morphology_class'] == cls]
            if areas:
                area_by_class[cls] = {
                    'mean': float(np.mean(areas)),
                    'std': float(np.std(areas)),
                    'min': float(np.min(areas)),
                    'max': float(np.max(areas)),
                    'median': float(np.median(areas))
                }

        # Shape metric summaries
        shape_metrics = {}
        for metric in ['circularity', 'aspect_ratio', 'solidity', 'eccentricity']:
            values = [c[metric] for c in all_classified if c[metric] is not None]
            if values:
                shape_metrics[metric] = {
                    'mean': float(np.mean(values)),
                    'std': float(np.std(values)),
                    'min': float(np.min(values)),
                    'max': float(np.max(values))
                }

        # Per-image statistics
        per_image_stats = []
        for img_idx, comps in images_dict.items():
            img_classified = MorphologyAnalyzer.classify_particles(comps)
            img_counts = {}
            for comp in img_classified:
                cls = comp['morphology_class']
                img_counts[cls] = img_counts.get(cls, 0) + 1

            per_image_stats.append({
                'image_index': int(img_idx) if img_idx != -1 else -1,
                'total_particles': len(img_classified),
                'class_counts': img_counts
            })

        return {
            'dataset': dataset_name,
            'total_particles': len(all_classified),
            'total_images': len(images_dict),
            'class_distribution': {k: int(v) for k, v in class_counts.items()},  # Convert to int
            'area_statistics': area_by_class,
            'shape_statistics': shape_metrics,
            'per_image_statistics': per_image_stats
        }



def process_single_dataset(name, image_dir, mask_dir):
    """Process a single dataset with error handling"""
    print(f"\n{'='*50}")
    print(f"Processing: {name}")

    # Load masks
    mask_dir = Path(mask_dir)
    mask_files = sorted(mask_dir.glob('*'))

    if not mask_files:
        print(f"   No files found in {mask_dir}")
        return None

    print(f" Found {len(mask_files)} mask files")

    masks = []
    filenames = []

    for mask_file in mask_files:
        mask = cv2.imread(str(mask_file), cv2.IMREAD_GRAYSCALE)
        if mask is not None:
            masks.append(mask)
            filenames.append(mask_file.name)
        else:
            print(f"    Could not read {mask_file.name}")

    print(f"   Successfully loaded {len(masks)} masks")

    # Generate GT counts
    print(f" Generating GT counts...")

    gt_counts = []
    cleaned_masks = []
    all_component_data = []

    for i, mask in enumerate(masks):
        cleaned, comp_data = preprocess_gt_mask(mask, image_idx=i)
        cleaned_masks.append(cleaned)
        gt_counts.append(count_particles(cleaned))

        # Add filename to each component
        for comp in comp_data:
            comp['filename'] = filenames[i]

        all_component_data.extend(comp_data)

    gt_counts = np.array(gt_counts)

    # Print statistics
    print(f"\n{'='*50}")
    print(f" {name} - Unified Microplastic Counting")
    print(f"{'='*50}")
    print(f"Images: {len(gt_counts):,}")
    print(f"Total particles: {gt_counts.sum():,}")
    print(f"Mean ± Std: {gt_counts.mean():.2f} ± {gt_counts.std():.2f}")
    print(f"Range: [{gt_counts.min()}, {gt_counts.max()}]")
    print(f"Median: {np.median(gt_counts):.1f}")

    # Morphology analysis
    morphology = MorphologyAnalyzer.analyze_dataset(name, all_component_data)

    if morphology:
        print(f"\n Particle Morphology (Post-hoc):")
        total = morphology['total_particles']
        for cls, count in morphology['class_distribution'].items():
            percentage = count / total * 100
            print(f"  {cls:<12}: {count:>5} particles ({percentage:5.1f}%)")

    # Save results
    output_dir = Path("gt_results") / name
    output_dir.mkdir(parents=True, exist_ok=True)

    # Save GT counts
    df_counts = pd.DataFrame({
        'filename': filenames,
        'gt_particle_count': gt_counts.astype(int),  # Convert to Python int
        'dataset': name
    })
    df_counts.to_csv(output_dir / "gt_counts.csv", index=False)

    # Save component data
    if all_component_data:
        # Convert all numpy types to Python types for JSON
        component_data_py = []
        for comp in all_component_data:
            comp_py = {}
            for key, value in comp.items():
                if isinstance(value, (np.integer, np.int32, np.int64)):
                    comp_py[key] = int(value)
                elif isinstance(value, (np.floating, np.float32, np.float64)):
                    comp_py[key] = float(value)
                elif isinstance(value, np.ndarray):
                    comp_py[key] = value.tolist()
                else:
                    comp_py[key] = value
            component_data_py.append(comp_py)

        df_components = pd.DataFrame(component_data_py)
        df_components.to_csv(output_dir / "component_data.csv", index=False)

    # Save morphology analysis
    if morphology:
        # Convert numpy types in morphology analysis
        morphology_py = json.loads(json.dumps(morphology, default=convert_numpy_types))
        with open(output_dir / "morphology_analysis.json", 'w') as f:
            json.dump(morphology_py, f, indent=2)

    print(f"\n Results saved to: {output_dir}")

    return {
        'name': name,
        'gt_counts': gt_counts,
        'filenames': filenames,
        'morphology': morphology,
        'output_dir': output_dir
    }

def convert_numpy_types(obj):
    """Convert numpy types to Python types for JSON serialization"""
    if isinstance(obj, (np.integer, np.int32, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_numpy_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(item) for item in obj]
    else:
        return obj



def run_simple_pipeline():
    """Simple pipeline that works"""
    print("="*70)
    print(" SIMPLE GT COUNTING PIPELINE")
    print("="*70)
    print(GTConfig.description())
    print()

    # Your dataset paths
    BASE_PATH = "/content/Full_Beads_Microplastic"

    datasets_config = [
        ("Train", f"{BASE_PATH}/training/training/Training_original",
                f"{BASE_PATH}/training/training/Finalmasks"),
        ("Validation", f"{BASE_PATH}/val/val/Original",
                     f"{BASE_PATH}/val/val/val_100masks"),
        ("Test", f"{BASE_PATH}/Testing/Testing/Original",
                f"{BASE_PATH}/Testing/Testing/testing_100masks")
    ]

    results = {}

    for name, img_dir, mask_dir in datasets_config:
        result = process_single_dataset(name, img_dir, mask_dir)
        if result:
            results[name] = result

    # Create summary report
    create_simple_report(results)

    return results

def create_simple_report(results):
    """Create simple summary report"""
    print(f"\n{'='*70}")
    print(" FINAL SUMMARY")
    print(f"{'='*70}")

    print(f"\n{'Dataset':<12} {'Images':<8} {'Particles':<10} {'Mean':<8} {'Beads %':<8}")
    print(f"{'-'*50}")

    total_images = 0
    total_particles = 0

    for name, data in results.items():
        counts = data['gt_counts']
        total_images += len(counts)
        total_particles += counts.sum()

        # Get bead percentage if available
        bead_pct = "—"
        if data.get('morphology'):
            total = data['morphology']['total_particles']
            bead_count = data['morphology']['class_distribution'].get('bead-like', 0)
            bead_pct = f"{bead_count/total*100:.1f}%" if total > 0 else "—"

        print(f"{name:<12} {len(counts):<8} {counts.sum():<10} {counts.mean():<8.1f} {bead_pct:<8}")

    print(f"\n{'Total':<12} {total_images:<8} {total_particles:<10} "
          f"{total_particles/total_images:<8.1f}")

    # Save combined results
    all_counts = []
    for name, data in results.items():
        df = pd.DataFrame({
            'filename': data['filenames'],
            'gt_particle_count': data['gt_counts'].astype(int),
            'dataset': name
        })
        all_counts.append(df)

    if all_counts:
        combined_df = pd.concat(all_counts, ignore_index=True)
        combined_df.to_csv("all_gt_counts.csv", index=False)
        print(f"\n Combined results saved to: all_gt_counts.csv")

    print(f"\n Your ground-truth counts are ready!")
    print(f"\n For your paper:")
    print(f"- Total: {total_particles:,} particles across {total_images:,} images")
    print(f"- Average: {total_particles/total_images:.1f} particles per image")
    print(f"- Methodology: {GTConfig.description()}")



if __name__ == "__main__":
    # Run the simplified pipeline
    results = run_simple_pipeline()

    print(f"\n{'='*70}")
    print(" PIPELINE COMPLETE")
    print(f"{'='*70}")

    print(f"\n Your results are in: gt_results/")
    print(f" To load and use GT counts:")

    print("\n# Load test set GT counts")
    print("test_df = pd.read_csv('gt_results/Test/gt_counts.csv')")
    print("test_gt_counts = test_df['gt_particle_count'].values")
    print("print(f'Test set: {len(test_gt_counts)} images, {test_gt_counts.sum()} particles')")

    print("\n# Evaluate your model")
    print("from sklearn.metrics import mean_absolute_error")
    print("mae = mean_absolute_error(test_gt_counts, your_model_predictions)")
    print("print(f'MAE: {mae:.2f} particles per image')")

 SIMPLE GT COUNTING PIPELINE
Ground-truth particle counts for mixed microplastics (beads and irregular particles) were derived from annotated segmentation masks by identifying connected components. Size (15–15000 pixels) and border (8-pixel exclusion) constraints were applied to remove noise and ambiguous edge particles. No shape-based filtering was applied at the ground-truth stage to maintain objectivity for irregular microplastics. Particle morphology was analyzed separately using post hoc shape descriptors.


Processing: Train
 Found 1381 mask files
   Successfully loaded 1381 masks
 Generating GT counts...

 Train - Unified Microplastic Counting
Images: 1,381
Total particles: 10,513
Mean ± Std: 7.61 ± 4.85
Range: [0, 31]
Median: 7.0

 Particle Morphology (Post-hoc):
  bead-like   :  8259 particles ( 78.6%)
  compact     :  1399 particles ( 13.3%)
  irregular   :   781 particles (  7.4%)
  fiber-like  :    74 particles (  0.7%)

 Results saved to: gt_results/Train

Processing: Vali

In [4]:
import pandas as pd
import numpy as np

# Load test set GT counts
test_df = pd.read_csv('gt_results/Test/gt_counts.csv')
test_gt_counts = test_df['gt_particle_count'].values
test_filenames = test_df['filename'].values

print(f"Test set: {len(test_gt_counts)} images")
print(f"Total particles: {test_gt_counts.sum()}")
print(f"Mean particles per image: {test_gt_counts.mean():.2f} ± {test_gt_counts.std():.2f}")

Test set: 139 images
Total particles: 802
Mean particles per image: 5.77 ± 3.16


In [10]:
import pandas as pd
import numpy as np

# Load test set GT counts
test_df = pd.read_csv('/content/gt_results/Train/gt_counts.csv')
test_gt_counts = test_df['gt_particle_count'].values
test_filenames = test_df['filename'].values

print(f"Train set: {len(test_gt_counts)} images")
print(f"Total particles: {test_gt_counts.sum()}")
print(f"Mean particles per image: {test_gt_counts.mean():.2f} ± {test_gt_counts.std():.2f}")

Train set: 1381 images
Total particles: 10513
Mean particles per image: 7.61 ± 4.85
